In [0]:
%pip install xgboost

In [0]:
dbutils.library.restartPython()

In [0]:
import os
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/default/flight_data/mllib_tmp"
dbutils.fs.mkdirs("/Volumes/workspace/default/flight_data/mllib_tmp")

import gc
import mlflow
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

FEATURE_TABLE = "gold_flight_features"
MODEL_VOLUME_PATH = "/Volumes/workspace/default/flight_data/models"
SEED = 42
print("Setup done.")

Setup done.


In [0]:
features = spark.table(FEATURE_TABLE).select(
    "label",
    "Reporting_Airline", "Origin", "DepTimeCategory",
    "Month", "DayOfWeek", "DepHour",
    "IsWeekend", "IsFixedHoliday", "IsHolidayWindow"
)

print(f"Total rows: {features.count():,}")
train_df, test_df = features.randomSplit([0.8, 0.2], seed=SEED)
print(f"Train: {train_df.count():,} | Test: {test_df.count():,}")

Total rows: 302,192
Train: 241,920 | Test: 60,272


In [0]:
categorical_cols = ["Reporting_Airline", "Origin", "DepTimeCategory"]
numeric_cols     = ["Month", "DayOfWeek", "DepHour", "IsWeekend", "IsFixedHoliday", "IsHolidayWindow"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in categorical_cols
]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in categorical_cols],
    outputCols=[f"{c}_ohe" for c in categorical_cols],
    handleInvalid="keep"
)

assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in categorical_cols] + numeric_cols,
    outputCol="features",
    handleInvalid="keep"
)

preprocessing_stages = indexers + [encoder, assembler]
print(f"Preprocessing stages: {len(preprocessing_stages)}")

Preprocessing stages: 5


In [0]:
auc_eval       = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
accuracy_eval  = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_eval        = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
precision_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
recall_eval    = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

def evaluate_model(model, test_data, model_name):
    predictions = model.transform(test_data)
    return {
        "model":     model_name,
        "roc_auc":   auc_eval.evaluate(predictions),
        "accuracy":  accuracy_eval.evaluate(predictions),
        "precision": precision_eval.evaluate(predictions),
        "recall":    recall_eval.evaluate(predictions),
        "f1":        f1_eval.evaluate(predictions),
    }, predictions

In [0]:
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50, regParam=0.01)
lr_pipeline = Pipeline(stages=preprocessing_stages + [lr])

with mlflow.start_run(run_name="LogisticRegression"):
    lr_model = lr_pipeline.fit(train_df)
    lr_metrics, _ = evaluate_model(lr_model, test_df, "LogisticRegression")
    for k, v in lr_metrics.items():
        if k != "model":
            mlflow.log_metric(k, v)

print(lr_metrics)
lr_model.write().overwrite().save(f"{MODEL_VOLUME_PATH}/logistic_regression_pipeline")

del lr_pipeline, lr_model
gc.collect()
print("LR saved, cache cleared.")

{'model': 'LogisticRegression', 'roc_auc': 0.6421816488327867, 'accuracy': 0.7583289089461109, 'precision': 0.7069352078448942, 'recall': 0.758328908946111, 'f1': 0.6544468026747908}
LR saved, cache cleared.


In [0]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=50,
    maxDepth=8,
    seed=SEED
)
rf_pipeline = Pipeline(stages=preprocessing_stages + [rf])

with mlflow.start_run(run_name="RandomForest"):
    rf_model = rf_pipeline.fit(train_df)
    rf_metrics, _ = evaluate_model(rf_model, test_df, "RandomForest")
    for k, v in rf_metrics.items():
        if k != "model":
            mlflow.log_metric(k, v)

print(rf_metrics)
rf_model.write().overwrite().save(f"{MODEL_VOLUME_PATH}/random_forest_pipeline")

del rf_pipeline, rf_model
gc.collect()
print("RF saved, cache cleared.")

{'model': 'RandomForest', 'roc_auc': 0.6599212116709646, 'accuracy': 0.7582957260419432, 'precision': 0.5750124081334776, 'recall': 0.7582957260419432, 'f1': 0.6540565385185508}
RF saved, cache cleared.


In [0]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxDepth=5,
    maxIter=30,
    seed=SEED
)
gbt_pipeline = Pipeline(stages=preprocessing_stages + [gbt])

with mlflow.start_run(run_name="GBT"):
    gbt_model = gbt_pipeline.fit(train_df)
    gbt_metrics, _ = evaluate_model(gbt_model, test_df, "GBT")
    for k, v in gbt_metrics.items():
        if k != "model":
            mlflow.log_metric(k, v)

print(gbt_metrics)
gbt_model.write().overwrite().save(f"{MODEL_VOLUME_PATH}/gbt_pipeline")

del gbt_pipeline, gbt_model
gc.collect()
print("GBT saved, cache cleared.")

{'model': 'GBT', 'roc_auc': 0.6853788452492768, 'accuracy': 0.7636049907087868, 'precision': 0.7251582069103726, 'recall': 0.7636049907087868, 'f1': 0.6863163918127411}
GBT saved, cache cleared.


In [0]:
phase2 = {
    "LogisticRegression": {"accuracy": 0.6182, "roc_auc": 0.6794, "f1": 0.4543},
    "RandomForest":       {"accuracy": 0.6454, "roc_auc": 0.7180, "f1": 0.4784},
    "XGBoost":            {"accuracy": 0.7076, "roc_auc": 0.7632, "f1": 0.5237},
}

phase3 = {
    "LogisticRegression": lr_metrics,
    "RandomForest":       rf_metrics,
    "GBT":                gbt_metrics,
}

rows = []
for p2_key, p3_key in [("LogisticRegression", "LogisticRegression"), ("RandomForest", "RandomForest"), ("XGBoost", "GBT")]:
    p2 = phase2[p2_key]
    p3 = phase3[p3_key]
    rows.append((
        f"{p2_key} (P2) vs {p3_key} (P3)" if p2_key != p3_key else p2_key,
        round(p2["accuracy"], 4), round(p3["accuracy"], 4),
        round(p2["roc_auc"], 4),  round(p3["roc_auc"], 4),
        round(p2["f1"], 4),       round(p3["f1"], 4),
    ))

comparison_df = spark.createDataFrame(
    rows,
    ["model", "p2_accuracy", "p3_accuracy", "p2_roc_auc", "p3_roc_auc", "p2_f1", "p3_f1"]
)
comparison_df.show(truncate=False)

comparison_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("phase_comparison")

+------------------------+-----------+-----------+----------+----------+------+------+
|model                   |p2_accuracy|p3_accuracy|p2_roc_auc|p3_roc_auc|p2_f1 |p3_f1 |
+------------------------+-----------+-----------+----------+----------+------+------+
|LogisticRegression      |0.6182     |0.7583     |0.6794    |0.6422    |0.4543|0.6544|
|RandomForest            |0.6454     |0.7583     |0.718     |0.6599    |0.4784|0.6541|
|XGBoost (P2) vs GBT (P3)|0.7076     |0.7636     |0.7632    |0.6854    |0.5237|0.6863|
+------------------------+-----------+-----------+----------+----------+------+------+

